# Event envelopes at 1.5σ and 2σ: implied equivalent depths

**Question:** Does defining stronger active envelope components produce cleaner
carrier fits and more concentrated apparent equivalent-depth distributions?

Daily 1990–2010 NOAA OLR and 34 Historical / 25 AMIP simulations use the same
filters, native calendars, preprocessing and raw active-component sinusoid fit.
The activity cutoff changes from 1.5σ to 2σ of the filtered OLR field; σ is
measured excluding 150 days at each record end. Active pixels are defined by the
**longitudinal Hilbert envelope**. Symmetry is projected over 20°S–20°N and
filtered OLR is averaged over northern 0–20° latitude before detecting components.

Each component contributes one fitted signed K/frequency pair and its resting-state,
branch-specific apparent equivalent depth. All candidates remain in the source
catalogs, including failed fits and undefined depths. No background model or
significance selection is used. [Reproduction and audit](../scripts/codex_sigma2_events/README.md).


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.paths import DATA_DIR

ROOT = DATA_DIR / "event_threshold_comparison"
WAVES = ["Kelvin", "n1ERW", "MRG", "n0EIG", "n1WIG"]
LABELS = {"Kelvin": "Kelvin", "n1ERW": "n=1 ER", "MRG": "MRG", "n0EIG": "n=0 EIG", "n1WIG": "n=1 WIG"}
SOURCES = [("observations", "NOAA", "k"), ("historical", "Historical model mean", "C0"), ("amip", "AMIP model mean", "C1")]
SIGMA_STYLES = {1.5: "-", 2.0: "--"}
KEYS = ["model", "experiment", "member", "wave"]
audit = json.loads((ROOT / "audit.json").read_text())
summary = pd.read_csv(ROOT / "catalog_summary.csv")
histograms = pd.read_csv(ROOT / "depth_histograms.csv.gz")
assert audit["status"] == "complete" and len(summary) == 600
assert not summary.duplicated(KEYS + ["detection_sigma"]).any()
display(summary.groupby(["experiment", "detection_sigma"]).agg(
    catalogs=("wave", "size"), candidates=("candidates", "sum"),
    valid_depths=("valid_depths", "sum"), screened_events=("screened_events", "sum")))


## NOAA: counts, fit support and implied depths

The valid-depth subset requires a successful fit and a positive depth on the
assigned dispersion branch. The screened subset additionally uses the unchanged
catalog criteria: no search-grid boundary, R² ≥ 0.3, at least half a sampled
spatial cycle and one temporal cycle. R² is computed on the component's active
pixels. Raising the activity threshold changes those pixels and the population.


In [ ]:
columns = ["candidates", "valid_depths", "screened_events", "screened_fraction",
           "median_fit_r2", "median_spatial_cycles", "median_temporal_cycles",
           "valid_h_median_m", "screened_h_median_m", "valid_h_above_200_fraction"]
noaa_table = summary.loc[summary.experiment.eq("observations")].set_index(["wave", "detection_sigma"])
display(noaa_table[columns].sort_index().round(3))


## Matched source/wave comparisons

Each row pairs the same model, experiment, member and wave family across
thresholds. Event IDs are never joined across thresholds: raising the cutoff
can split, shorten or remove a component. The table reports NOAA's direct
changes and model-wise median changes for the two model ensembles.


In [ ]:
baseline = summary.loc[summary.detection_sigma.eq(1.5)].set_index(KEYS)
stronger = summary.loc[summary.detection_sigma.eq(2.)].set_index(KEYS).reindex(baseline.index)
assert len(baseline) == len(stronger) == 300
paired = pd.DataFrame(index=baseline.index)
for column in ["candidates", "valid_depths", "screened_events"]:
    paired[column + "_change_pct"] = 100 * (stronger[column] / baseline[column].replace(0, np.nan) - 1)
paired["screened_fraction_change_pp"] = 100 * (stronger.screened_fraction - baseline.screened_fraction)
paired["median_r2_change"] = stronger.median_fit_r2 - baseline.median_fit_r2
paired["median_temporal_cycles_change"] = stronger.median_temporal_cycles - baseline.median_temporal_cycles
paired["valid_h_median_change_m"] = stronger.valid_h_median_m - baseline.valid_h_median_m
display(paired.groupby(["experiment", "wave"]).median().round(3))


## Depth distributions: all successful, dispersion-valid fits

Each curve counts events, with 5 m bins. Densities divide by **all** selected
events, including those above the 200 m display limit, so a curve need not
integrate to one within the plot. The light gray band marks 10–80 m without
restricting detection or fitting. Each model is normalized separately before
forming an equally weighted ensemble mean. Solid lines use 1.5σ; dashed lines
use 2σ. Missing depth estimates are counted in the audit tables above.


In [ ]:
selection_title = "all successful, dispersion-valid fits"
selection = 'valid'
for wave in WAVES:
    fig, ax = plt.subplots(figsize=(8, 4.7))
    for experiment, source_label, color in SOURCES:
        for sigma, linestyle in SIGMA_STYLES.items():
            part = histograms.loc[
                histograms.experiment.eq(experiment) & histograms.wave.eq(wave)
                & histograms.detection_sigma.eq(sigma) & histograms.selection.eq(selection)
                & histograms.selected_events.gt(0)
            ]
            if part.empty:
                print(f"No {selection} depths: {experiment}, {wave}, {sigma} sigma")
                continue
            density = part.groupby("lower_m").density_per_m.mean().sort_index()
            model_count = part[["model", "member"]].drop_duplicates().shape[0]
            label = f"{source_label}, {sigma:g}σ"
            if experiment != "observations":
                label += f" (n={model_count})"
            ax.stairs(density, np.arange(0., 200.1, 5.), baseline=None,
                      color=color, linestyle=linestyle, linewidth=1.8, label=label)
    ax.axvspan(10, 80, color=".94", zorder=-2)
    ax.set(xlim=(0, 200), ylim=(0, None), xlabel="Apparent equivalent depth [m]",
           ylabel="Fraction of selected events per metre [m$^{-1}$]",
           title=f"{LABELS[wave]}: {selection_title}")
    ax.legend(fontsize=8)
    ax.grid(alpha=.2)
    fig.tight_layout()
    plt.show()


## Depth distributions with the existing fit-quality screen

This additional view keeps the same screen at both thresholds. Its denominator
is each record's full screened population, including depths above 200 m. The
legend states the number of contributing models; a source with no screened
events has no defined normalized distribution.


In [ ]:
selection_title = "existing fit-quality subset"
selection = 'screened'
for wave in WAVES:
    fig, ax = plt.subplots(figsize=(8, 4.7))
    for experiment, source_label, color in SOURCES:
        for sigma, linestyle in SIGMA_STYLES.items():
            part = histograms.loc[
                histograms.experiment.eq(experiment) & histograms.wave.eq(wave)
                & histograms.detection_sigma.eq(sigma) & histograms.selection.eq(selection)
                & histograms.selected_events.gt(0)
            ]
            if part.empty:
                print(f"No {selection} depths: {experiment}, {wave}, {sigma} sigma")
                continue
            density = part.groupby("lower_m").density_per_m.mean().sort_index()
            model_count = part[["model", "member"]].drop_duplicates().shape[0]
            label = f"{source_label}, {sigma:g}σ"
            if experiment != "observations":
                label += f" (n={model_count})"
            ax.stairs(density, np.arange(0., 200.1, 5.), baseline=None,
                      color=color, linestyle=linestyle, linewidth=1.8, label=label)
    ax.axvspan(10, 80, color=".94", zorder=-2)
    ax.set(xlim=(0, 200), ylim=(0, None), xlabel="Apparent equivalent depth [m]",
           ylabel="Fraction of selected events per metre [m$^{-1}$]",
           title=f"{LABELS[wave]}: {selection_title}")
    ax.legend(fontsize=8)
    ax.grid(alpha=.2)
    fig.tight_layout()
    plt.show()


## Interpretation

A higher activity cutoff can increase R² while leaving fewer sampled cycles
to constrain propagation. Assess R², support, boundary flags, the fraction with
defined depth, and the depth distributions together. The depth estimates assume
zero background wind and the assigned wave branch. A narrower histogram or a
higher R² alone does not establish more accurate physical depths.

Full threshold-specific catalogs are identified by `catalog` and `run_id` in
`catalog_summary.csv`; all candidates remain available for further inspection.
No standalone figure files are written by this notebook.
